# BJ-235 — Beijing native line and 3/1/0 SP time series

Contract: `BJ-UNIFIED-BJ-SP-TIMESERIES-001 / v1` with PI amendment v2 (2026-08-17).

The primary historical delivery is the explicit `RETROSPECTIVE_RELATIVE_PROXY`
M×3 panel at initial/10h/1h/final buckets. It does not fabricate exact clocks
and must not be labelled unrecoverable solely for lacking them.


## 0. Environment and binding

This notebook is executed from the delivery worktree; it reads the canonical SQLite store built by `scripts/bj235_build.py` and the recoverability report it produced. Data root points at the external data store (gitignored).

In [1]:
import json, os, sys, sqlite3
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
CANON = Path(os.environ.get('BJ235_CANONICAL', ROOT / 'data' / 'derived' / 'bj235' / 'bj_sp_canonical.sqlite'))
OUT = Path(os.environ.get('BJ235_OUT', ROOT / 'results' / 'bj235_bj_sp_timeseries_v2'))
report_path = OUT / 'RECOVERABILITY_REPORT.json'
con = sqlite3.connect(f'file:{CANON}?mode=ro', uri=True)
con.row_factory = sqlite3.Row
report = json.loads(report_path.read_text(encoding='utf-8'))
print('canonical db:', CANON)
print('report loaded:', report_path)


canonical db: /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_296eb0dc/data/derived/bj235_v2/bj_sp_canonical.sqlite
report loaded: /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_296eb0dc/results/bj235_bj_sp_timeseries_v2/RECOVERABILITY_REPORT.json


## 1. Source registry (REQ-BJ01: every accepted quote binds source + clock)

Registered in the canonical store with admissible-use class and clock semantics per #234.

In [2]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)
src = pd.read_sql_query("SELECT source_id, provider, clock_semantics, admissible_use, first_usable_at_utc FROM source_registry", con)
src

,source_id,provider,clock_semantics,admissible_use,first_usable_at_utc
0,beidan_live_500,500.com BJDC bet page,trusted_receive,STRICT_CAUSAL,2026-05-02T00:57:44Z
1,bjdc_500_xml_34,500.com BJDC live SP XML (playid 34 handicap_3way),native_source_plus_trusted_receive,STRICT_CAUSAL,2026-06-29T16:47:31Z
2,beidan_xlsm_archive,internal Excel VBA archive (beidan_xlsm),label_proxy_initial_10h_1h_final,RETROSPECTIVE_PROXY,2014-01-01T00:00:00Z
3,nowscore_history,cp.nowscore.com BJDC SP history,native_source_change_time,RETROSPECTIVE_PROXY,NaN
4,okooo_prekick,okooo BJDC prekick archive,native_source_change_time,RETROSPECTIVE_PROXY,NaN
5,beidan_history,beidan_history.db settlement rows,settlement_time,SETTLEMENT_ONLY,2024-01-01T00:00:00Z
6,official_500_settlements,500.com kaijiang settlements,settlement_time,SETTLEMENT_ONLY,2014-01-01T00:00:00Z
7,beidan_features,beidan_features.db EU probability features,no_observation_clock,UNUSABLE,NaN


## 2. Raw presence waterfall (REQ-BJ06)

From raw source rows through identity join, line state, timestamp gate, complete triplet and final causal intersection.

In [3]:
wf = report["waterfall_raw"]
ct = report["complete_triplets"]
print("RAW PRESENCE (source rows):")
for k, v in wf.items():
    print(f"  {k:32s} {v:>12,}")
print()
print("COMPLETE 3/1/0 TRIPLETS (strict causal observations):")
for k, v in ct.items():
    print(f"  {k:32s} {v:>12,}")
print()
print("AS-OF REPLAY STATS:")
for k, v in report["build_steps"].get("asof", {}).items():
    print(f"  {k:32s} {v:>12,}")

RAW PRESENCE (source rows):
  beidan_live_500_raw                         0
  bjdc_500_xml_34_raw                         0
  xlsm_label_snapshots_51M           51,551,016
  history_settlement_rows                     0

COMPLETE 3/1/0 TRIPLETS (strict causal observations):
  beidan_live_500                             0
  bjdc_500_xml_34                             0
  total_strict_causal_obs                     0

AS-OF REPLAY STATS:


## 3. Issue-level support by cohort (REQ-BJ06/REQ-BJ08)

Strict historical (2014–2025), retrospective proxy, and prospective (2026-05+) reported separately.

In [4]:
support = report['issue_support']
print('ISSUE-LEVEL SUPPORT BY COHORT:')
for cohort, values in support['by_cohort'].items():
    print(' ', cohort, values)
pd.DataFrame(support['per_issue']).tail(15)


ISSUE-LEVEL SUPPORT BY COHORT:
  strict_historical {'issues': 701, 'fixtures_total': 192709, 'fixtures_complete_asof': 0}
  retrospective_proxy {'issues': 701, 'fixtures_total': 192709, 'fixtures_present': 192702}


,issue_code,year,cohort,fixtures_total,fixtures_present,fixtures_complete_asof
1387,25024,2025,strict_historical,80,NaN,0.0
1388,25031,2025,retrospective_proxy,354,354.0,NaN
1389,25031,2025,strict_historical,354,NaN,0.0
1390,25032,2025,retrospective_proxy,413,413.0,NaN
1391,25032,2025,strict_historical,413,NaN,0.0
1392,25033,2025,retrospective_proxy,407,407.0,NaN
1393,25033,2025,strict_historical,407,NaN,0.0
1394,25034,2025,retrospective_proxy,116,116.0,NaN
1395,25034,2025,strict_historical,116,NaN,0.0
1396,25035,2025,retrospective_proxy,369,369.0,NaN


## 4. Historical proxy panel and strict-clock boundary (REQ-BJ06/BJ08/BJ09)

The strict-clock historical subset remains zero. The separately labelled relative
proxy panel is a positive delivery; it has source/issue/fixture/native-line/outcome/
bucket/availability provenance and never substitutes settlement or later data.


In [5]:
hist = report['historical']
panel = hist['panel']
print('DISPOSITION:', hist['disposition'])
print('strict historical causal support:', hist['strict_historical_causal_support'])
print('proxy cells / present / missing:', panel['cells_total'], panel['cells_present'], panel['cells_missing'])
print('proxy issues / fixtures:', panel['issues_covered'], panel['fixtures_covered'])
print('buckets:', panel['buckets_profiled'])
pd.DataFrame([{'snapshot_bucket': key, **value} for key, value in hist['historical_panel'].items()])


DISPOSITION: HISTORICAL_RELATIVE_PROXY_PANEL
strict historical causal support: 0
proxy cells / present / missing: 2312508 2309166 3342
proxy issues / fixtures: 701 192709
buckets: ['10h', '1h', 'final', 'initial']


,snapshot_bucket,issues,fixtures,cells,cells_present,cells_missing
0,10h,701,192709,578127,577035,1092
1,1h,701,192709,578127,577332,795
2,final,701,192709,578127,577197,930
3,initial,701,192709,578127,577602,525


## 5. Line state retention (REQ-BJ04)

Handicap lines are preserved per observation; as-of selection never joins across different native-line states.

In [6]:
line_q = """
SELECT mo.handicap_line, COUNT(*) n_obs, COUNT(DISTINCT mo.fixture_id) n_fixtures
FROM market_observation mo
WHERE mo.source_id='bjdc_500_xml_34'
GROUP BY mo.handicap_line ORDER BY n_obs DESC
"""
lines = pd.read_sql_query(line_q, con)
lines

# line states per fixture: how many fixtures ever had >1 distinct line observed
multi = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT fixture_id FROM market_observation
        WHERE source_id='bjdc_500_xml_34' AND handicap_line IS NOT NULL
        GROUP BY fixture_id HAVING COUNT(DISTINCT handicap_line) > 1
    )
""").fetchone()[0]
print("fixtures with >1 observed native-line state:", multi)

fixtures with >1 observed native-line state: 0


## 6. Probability separation (REQ-BJ05)

Raw SP, normalized inverse-SP probability and winning/final SP are physically separate tables. No `0.65`, no settlement SP in inference tables.

In [7]:
pe = report['build_steps'].get('probabilities', {})
print('probability rows built:', pe.get('probability_rows', 0))
bad_065 = con.execute("SELECT COUNT(*) FROM market_quote WHERE decimal_odds=0.65").fetchone()[0]
panel_065 = con.execute("SELECT COUNT(*) FROM relative_horizon_panel WHERE decimal_odds=0.65").fetchone()[0]
print('0.65 quote rows:', bad_065, 'panel rows:', panel_065)


probability rows built: 0
0.65 quote rows: 0 panel rows: 0


## 7. Exclusion ledger (REQ-BJ02: incomplete observations remain explicit)

Non-causal rows are not silently dropped — they are registered with reason codes.

In [8]:
excl = pd.read_sql_query("""
    SELECT reason_code, COUNT(*) n, SUM(raw_count) total_raw
    FROM exclusion_ledger GROUP BY reason_code ORDER BY n DESC
""", con)
excl

,reason_code,n,total_raw
0,NO_OBSERVATION,1,2560


## 8. Package integrity and common v2 handoff

The immutable package includes the canonical table export, the common `relative_horizon_view`
long M×3 parquet required by #238, a manifest, and SHA-256 coverage.


In [9]:
manifest = json.loads((OUT / "MANIFEST.json").read_text(encoding="utf-8"))
print("package:", manifest["package"])
for name, t in manifest["tables"].items():
    print(f"  {name:24s} {t['rows']:>12,} rows  {t['file']}")
sums = json.loads((OUT / "SHA256SUMS.json").read_text(encoding="utf-8"))
print("hash-bound files:", len(sums))

package: bj235_bj_sp_timeseries_v2
  source_registry                     8 rows  source_registry.parquet
  ingestion_batch                     1 rows  ingestion_batch.parquet
  fixture                       192,709 rows  fixture.parquet
  market_observation                  0 rows  market_observation.parquet
  market_quote                        0 rows  market_quote.parquet
  decision_epoch                      0 rows  decision_epoch.parquet
  as_of_selection                     0 rows  as_of_selection.parquet
  probability_estimate                0 rows  probability_estimate.parquet
  settlement                          0 rows  settlement.parquet
  exclusion_ledger                    1 rows  exclusion_ledger.parquet
  relative_horizon_panel      2,312,508 rows  relative_horizon_panel.parquet
  relative_horizon_view       2,312,508 rows  relative_horizon_view.parquet
hash-bound files: 15


## 9. Acceptance trace

| REQ | Evidence |
|---|---|
| BJ01 | source registry plus source/issue/fixture/native-line/outcome/bucket/provenance hash |
| BJ02 | finite positive quotes or explicit `availability=missing` cells |
| BJ03 | strict as-of replay remains separate and rejects post-decision/post-kickoff rows |
| BJ04 | native line is bound to every panel cell; no cross-line replay join |
| BJ05 | no settlement fields or 0.65 in inference/panel view |
| BJ06 | strict historical, retrospective proxy, and prospective support are separate |
| BJ07 | deterministic panel ids and idempotent `INSERT OR IGNORE` ingestion |
| BJ08 | positive `HISTORICAL_RELATIVE_PROXY_PANEL` disposition with limits |
| BJ09 | issue/horizon coverage and `relative_horizon_view.parquet` |
